# PaySim Fraud Detection: Logistic Regression to a Pruned Decision Tree

This notebook connects the M13 Logistic Regression and M14 Decision Tree concepts into one modeling journey:

`baseline -> probability and cross-entropy -> regularization -> threshold tuning -> entropy tree -> overfitting -> pruning -> business-focused comparison`

The target is `isFraud`. The notebook uses the PaySim schema and evaluates precision, recall, F1, ROC-AUC, and PR-AUC rather than accuracy alone.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, log_loss, precision_recall_curve,
                             RocCurveDisplay, PrecisionRecallDisplay)

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')

In [ ]:
# Expected input: data/paysim.csv
df = pd.read_csv('data/paysim.csv')
expected_columns = ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg',
                    'newbalanceOrig', 'nameDest', 'oldbalanceDest',
                    'newbalanceDest', 'isFraud', 'isFlaggedFraud']
missing = sorted(set(expected_columns) - set(df.columns))
if missing:
    raise ValueError(f'Missing expected columns: {missing}')

print(df.shape)
display(df.head())
display(df.dtypes.to_frame('dtype'))
display(df['isFraud'].value_counts(normalize=True).rename('fraction'))

## 1. EDA and modeling choices

Fraud is usually the minority class. `nameOrig` and `nameDest` are identifiers, so using their raw values would encourage memorization rather than generalizable behavior. We keep transaction and balance features, plus one-hot encode `type`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x='type', hue='isFraud', ax=axes[0])
axes[0].set_title('Transactions by type and fraud label')
fraud_rate = df.groupby('type', observed=True)['isFraud'].mean().sort_values(ascending=False)
fraud_rate.plot.bar(ax=axes[1], color='crimson')
axes[1].set_title('Fraud rate by transaction type')
axes[1].set_ylabel('Fraud rate')
plt.tight_layout()

In [ ]:
numeric_features = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
                    'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud']
categorical_features = ['type']
X = df[numeric_features + categorical_features].copy()
y = df['isFraud'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

preprocess = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

## 2. Logistic Regression baseline

Logistic Regression maps a linear score through the sigmoid function: `p = 1 / (1 + exp(-z))`. Its training objective is log loss, also called binary cross-entropy. The probability is useful for ranking risk and for changing the operating threshold.

L1 regularization can drive weak coefficients to zero; L2 generally shrinks coefficients without selecting as aggressively. In scikit-learn, smaller `C` means stronger regularization.

In [ ]:
logistic_l2 = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(penalty='l2', C=1.0, class_weight='balanced',
                                  solver='liblinear', max_iter=1000, random_state=RANDOM_STATE))
])
logistic_l2.fit(X_train, y_train)
p_logistic = logistic_l2.predict_proba(X_test)[:, 1]

def evaluate_model(name, y_true, probability, threshold=0.50):
    prediction = (probability >= threshold).astype(int)
    return {'model': name, 'threshold': threshold,
            'accuracy': accuracy_score(y_true, prediction),
            'precision': precision_score(y_true, prediction, zero_division=0),
            'recall': recall_score(y_true, prediction, zero_division=0),
            'f1': f1_score(y_true, prediction, zero_division=0),
            'roc_auc': roc_auc_score(y_true, probability),
            'pr_auc': average_precision_score(y_true, probability),
            'log_loss': log_loss(y_true, probability)}

baseline_result = evaluate_model('Logistic Regression L2', y_test, p_logistic)
display(pd.DataFrame([baseline_result]))
print(classification_report(y_test, (p_logistic >= .50).astype(int), zero_division=0))

In [ ]:
# Compare L1 and L2 regularization. This is a modeling comparison, not a claim that L1 is always better.
regularization_results = []
for penalty, solver in [('l1', 'liblinear'), ('l2', 'liblinear')]:
    model = Pipeline([('preprocess', preprocess),
                      ('model', LogisticRegression(penalty=penalty, C=1.0,
                                                    class_weight='balanced', solver=solver,
                                                    max_iter=1000, random_state=RANDOM_STATE))])
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    regularization_results.append(evaluate_model(f'Logistic Regression {penalty.upper()}', y_test, p))
display(pd.DataFrame(regularization_results))

## 3. Threshold optimization

A threshold is a business decision layered on top of the probability model. Lowering it generally catches more fraud but sends more legitimate transactions to decline or review. We inspect the trade-off rather than assuming `0.50` is correct.

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.05)
threshold_results = pd.DataFrame([
    evaluate_model('Logistic Regression', y_test, p_logistic, t)
    for t in thresholds
])
display(threshold_results[['threshold', 'precision', 'recall', 'f1', 'pr_auc']])
threshold_results.set_index('threshold')[['precision', 'recall', 'f1']].plot(figsize=(9, 4), marker='o')
plt.title('Fraud precision/recall trade-off by threshold')
plt.ylabel('score')
plt.show()

## 4. Decision Tree with entropy

A tree recursively chooses splits that reduce impurity. With `criterion='entropy'`, the split search is based on entropy and weighted child entropy. A pure node contains one class; an impure node contains a mixture. Trees represent nonlinear interactions directly, but an unrestricted tree can memorize the training set.

In [ ]:
tree_unrestricted = Pipeline([
    ('preprocess', preprocess),
    ('model', DecisionTreeClassifier(criterion='entropy', class_weight='balanced',
                                     random_state=RANDOM_STATE))
])
tree_unrestricted.fit(X_train, y_train)
p_tree = tree_unrestricted.predict_proba(X_test)[:, 1]

train_p_tree = tree_unrestricted.predict_proba(X_train)[:, 1]
overfit_check = pd.DataFrame([
    evaluate_model('Unrestricted tree - train', y_train, train_p_tree),
    evaluate_model('Unrestricted tree - test', y_test, p_tree)
])
display(overfit_check)
tree_estimator = tree_unrestricted.named_steps['model']
print('depth:', tree_estimator.get_depth(), 'leaves:', tree_estimator.get_n_leaves())

In [ ]:
# Inspect a shallow tree as human-readable if/then logic.
feature_names = tree_unrestricted.named_steps['preprocess'].get_feature_names_out()
small_tree = DecisionTreeClassifier(criterion='entropy', max_depth=3,
                                     class_weight='balanced', random_state=RANDOM_STATE)
X_train_encoded = tree_unrestricted.named_steps['preprocess'].transform(X_train)
small_tree.fit(X_train_encoded, y_train)
print(export_text(small_tree, feature_names=list(feature_names), decimals=2))

plt.figure(figsize=(18, 8))
plot_tree(small_tree, feature_names=feature_names, class_names=['legitimate', 'fraud'],
          filled=True, rounded=True, proportion=True, max_depth=3)
plt.title('Entropy decision tree: first three levels')
plt.show()

## 5. Cost-complexity pruning

`cost_complexity_pruning_path()` supplies candidate `ccp_alpha` values. Increasing `ccp_alpha` penalizes tree complexity and removes branches. We choose the value using validation PR-AUC, while also inspecting recall, precision, F1, depth, and leaf count.

In [ ]:
# Use a validation split inside the training data so the test set remains a final check.
X_fit, X_validation, y_fit, y_validation = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=RANDOM_STATE
)
prep_for_path = preprocess.fit(X_fit, y_fit)
X_fit_encoded = prep_for_path.transform(X_fit)
X_validation_encoded = prep_for_path.transform(X_validation)

path_tree = DecisionTreeClassifier(criterion='entropy', class_weight='balanced', random_state=RANDOM_STATE)
path_tree.fit(X_fit_encoded, y_fit)
path = path_tree.cost_complexity_pruning_path(X_fit_encoded, y_fit)
ccp_alphas = path.ccp_alphas
impurities = path.impurities

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(ccp_alphas, impurities, marker='o', drawstyle='steps-post')
ax[0].set(xlabel='ccp_alpha', ylabel='total leaf impurity', title='Pruning path')
ax[1].plot(ccp_alphas, [DecisionTreeClassifier(criterion='entropy', class_weight='balanced',
        ccp_alpha=a, random_state=RANDOM_STATE).fit(X_fit_encoded, y_fit).get_depth() for a in ccp_alphas],
        marker='o')
ax[1].set(xlabel='ccp_alpha', ylabel='depth', title='Complexity decreases as alpha grows')
plt.tight_layout()

In [ ]:
pruned_results = []
pruned_models = []
for alpha in ccp_alphas[:-1]:
    model = DecisionTreeClassifier(criterion='entropy', class_weight='balanced',
                                  ccp_alpha=alpha, random_state=RANDOM_STATE)
    model.fit(X_fit_encoded, y_fit)
    probability = model.predict_proba(X_validation_encoded)[:, 1]
    row = evaluate_model('Pruned entropy tree', y_validation, probability, threshold=0.30)
    row.update({'ccp_alpha': alpha, 'depth': model.get_depth(), 'leaves': model.get_n_leaves()})
    pruned_results.append(row)
    pruned_models.append(model)

pruned_results = pd.DataFrame(pruned_results).sort_values('pr_auc', ascending=False)
display(pruned_results.head(10))
best_row = pruned_results.iloc[0]
best_alpha = float(best_row['ccp_alpha'])
print(f'Chosen alpha={best_alpha:.8g}; validation PR-AUC={best_row.pr_auc:.4f}; depth={int(best_row.depth)}; leaves={int(best_row.leaves)}')

In [ ]:
# Refit the selected alpha on all training rows, then evaluate once on untouched test data.
preprocess_final = preprocess.fit(X_train, y_train)
X_train_final = preprocess_final.transform(X_train)
X_test_final = preprocess_final.transform(X_test)
final_tree_estimator = DecisionTreeClassifier(criterion='entropy', class_weight='balanced',
                                               ccp_alpha=best_alpha, random_state=RANDOM_STATE)
final_tree_estimator.fit(X_train_final, y_train)
p_final_tree = final_tree_estimator.predict_proba(X_test_final)[:, 1]

comparison = pd.DataFrame([
    evaluate_model('Logistic Regression L2', y_test, p_logistic, threshold=0.30),
    evaluate_model('Unrestricted entropy tree', y_test, p_tree, threshold=0.30),
    evaluate_model('Pruned entropy tree', y_test, p_final_tree, threshold=0.30)
])
display(comparison)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
RocCurveDisplay.from_predictions(y_test, p_logistic, name='Logistic Regression', ax=axes[0])
RocCurveDisplay.from_predictions(y_test, p_final_tree, name='Pruned tree', ax=axes[0])
PrecisionRecallDisplay.from_predictions(y_test, p_logistic, name='Logistic Regression', ax=axes[1])
PrecisionRecallDisplay.from_predictions(y_test, p_final_tree, name='Pruned tree', ax=axes[1])
plt.tight_layout()

## Conclusion

I started with Logistic Regression because it provides an interpretable probability baseline and exposes the effects of regularization and threshold selection. I then evaluated an entropy-based Decision Tree because fraud can depend on nonlinear feature interactions. The unrestricted tree was checked for overfitting, and cost-complexity pruning generated simpler candidates through `ccp_alpha`.

The final model should be selected from validation and test evidence. For fraud detection, recall matters because missed fraud is costly, while precision matters because false positives create unnecessary declines or reviews. PR-AUC is especially useful when fraud is rare. The tree does not automatically replace Logistic Regression; it earns its place only if its validation and business trade-offs are better.